In [1]:
import os
import torch
from dotenv import load_dotenv
from crewai import LLM, Agent, Task, Crew, Process
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from crewai.tools import tool

load_dotenv()

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize LLMs (Using Groq as per your LegalRAG2 setup)
legal_llm = LLM(model="groq/llama-3.3-70b-versatile")
research_llm = LLM(model="groq/llama-3.1-8b-instant")

# Set dummy key to satisfy any internal Pydantic checks (Method from Lawglance)
os.environ["OPENAI_API_KEY"] = "NA"

In [2]:
# 1. Initialize Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={'device': device}
)

# 2. Connect to your existing Vector Database
# Update this path to your actual vector_database folder
db_path = "C:/Users/user/Desktop/RAG Projects/Legal RAG 1/vector_database"
vector_store = Chroma(persist_directory=db_path, embedding_function=embeddings)

# 3. Create the Custom Tool function
@tool("legal_research_tool")
def legal_research_tool(query: str):
    """
    Searches the official legal vector database. 
    Use this tool to find specific statutes, constitutional articles, and case law relevant to the user's query.
    """
    # Using similarity search logic from your chains.py
    docs = vector_store.similarity_search(query, k=5)
    
    # Combine results into a single string for the agent to read
    return "\n\n".join([f"Source Content: {doc.page_content}" for doc in docs])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# Agent 1: The Researcher
researcher = Agent(
    role='Legal Researcher',
    goal='Retrieve exact legal provisions and citations related to {user_query}',
    backstory='Expert in Indian Constitutional law and criminal statutes. Skilled at finding the needle in the legal haystack.',
    tools=[legal_research_tool],
    llm=research_llm,
    verbose=True
)

# Agent 2: The Strategist
strategist = Agent(
    role='Legal Strategist',
    goal='Analyze the findings from the researcher and determine the legal hierarchy (e.g., Central vs. State law)',
    backstory='Senior legal counsel specializing in jurisdictional conflicts and constitutional interpretation.',
    llm=legal_llm,
    verbose=True
)

# Agent 3: The Advisor
advisor = Agent(
    role='Legal Advisor',
    goal='Synthesize a final, easy-to-understand response for the user based on the strategist\'s analysis',
    backstory='A compassionate legal advisor who translates complex legalese into actionable advice for citizens.',
    llm=legal_llm,
    verbose=True
)

In [4]:
research_task = Task(
    description='Search the database for all legal articles or sections relevant to: {user_query}. Provide direct quotes.',
    expected_output='A list of specific legal sections and their content.',
    agent=researcher
)

strategy_task = Task(
    description='Based on the research, explain the legal implications. If there is a conflict between laws, explain which one prevails.',
    expected_output='A strategic analysis of the legal hierarchy and applicable rules.',
    agent=strategist
)

advisory_task = Task(
    description='Summarize the strategy into a final answer for the user. Be concise and professional.',
    expected_output='A final response that directly answers the user\'s original question.',
    agent=advisor
)

In [5]:
legal_crew = Crew(
    agents=[researcher, strategist, advisor],
    tasks=[research_task, strategy_task, advisory_task],
    process=Process.sequential,
    memory=False, # This is CRITICAL to avoid the Pydantic/Embedder error
    verbose=True
)

In [6]:
# Execute
user_input = "If a state law conflicts with a central law in India, which one prevails?"
result = legal_crew.kickoff(inputs={"user_query": user_input})

print("\n" + "="*30)
print("FINAL LEGAL ADVICE:")
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  806982e3-9173-4647-a99f-0c4402b0ceae                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the database for all legal articles or sections relevant to: If a state law conflicts with a      │
│  central law in India, which one prevails?. Provide direct quotes.                                              │
│  ID: 6972771d-c367-4f35-88e7-f5cefd1dac59                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Task: Search the database for all legal articles or sections relevant to: If a state law conflicts with a      │
│  central law in India, which one prevails?. Provide direct quotes.                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: legal_research_tool                                                                                      │
│  Args: {'query': 'If a state law conflicts with a central law in India, which one prevails?'}                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool legal_research_tool executed with result: ...


...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Summarize the strategy into a final answer for the user. Be concise and professional.                          │
│  Agent:                                                                                                         │
│  Legal Advisor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  806982e3-9173-4647-a99f-0c4402b0ceae                                                                           │
│  Final Output: The legal research tool found the following relevant sections and their content:                 │
│                                                                                                                 │
│   Article 246(2) of the Constitution of India:                                                                  │
│  "The Parliament has exclusive power to make laws with respect to any matter not enumerated in the Concurrent   │
│  List or the State List."                                                                                       │
│                                                                                                                 │
│   Article 246(3) of the Constitution of India:                                                                  │
│  "The power to make laws on a matter in the Concurrent List shall be exercised by the Parliament and, subject   │
│  to the provisions of clause (2), the competence of the Legislature of a State to make laws in relation to      │
│  such matter in the State List is restricted to the extent to which the competent Legislature can make laws in  │
│  the area of the Concurrent List."                                                                              │
│                                                                                                                 │
│   Entry 52 of the State List (VII):                                                                             │
│  "Laws and procedures regulating the business of money-lending."                                                │
│                                                                                                                 │
│   Entry 97 of the Concurrent List (XIV):                                                                        │
│  "Trade marks and merchandise marks."                                                                           │
│                                                                                                                 │
│   Entry 109 of the Concurrent List (XIV):                                                                       │
│  "Trade and commerce in, and the production, supply and distribution of, certain goods."                        │
│                                                                                                                 │
│   Article 254 of the Constitution of India:                                                                     │
│  (1) If any provision of a State law is repugnant to any provision of a law made by Parliament which            │
│  Parliament is competent to enact, notwithstanding anything in sub-section (1) of section 204, the law made by  │
│  Parliament, whether passed before or after the State law, shall, subject to any provision made by Parliament   │
│  under article 249 or article 250, prevail and the provision of the State law shall, to the extent of           │
│  repugnancy, be void.                                                                                           │
│                                                       


FINAL LEGAL ADVICE:
The legal research tool found the following relevant sections and their content:

 Article 246(2) of the Constitution of India:
"The Parliament has exclusive power to make laws with respect to any matter not enumerated in the Concurrent List or the State List."

 Article 246(3) of the Constitution of India:
"The power to make laws on a matter in the Concurrent List shall be exercised by the Parliament and, subject to the provisions of clause (2), the competence of the Legislature of a State to make laws in relation to such matter in the State List is restricted to the extent to which the competent Legislature can make laws in the area of the Concurrent List."

 Entry 52 of the State List (VII):
"Laws and procedures regulating the business of money-lending."

 Entry 97 of the Concurrent List (XIV):
"Trade marks and merchandise marks."

 Entry 109 of the Concurrent List (XIV):
"Trade and commerce in, and the production, supply and distribution of, certain goods."

 A

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:
# Question 1
user_input = "I was caught jumping a red light and the police officer is asking for a 5,000 rupee fine. Is this amount correct?"
result = legal_crew.kickoff(inputs={"user_query": user_input})

# Verification Code
print(f"Evaluation: {'PASS' if 'Motor Vehicles Act' in result.raw else 'FAIL'}")
print(f"Reasoning: Check if the AI corrected the fine amount based on the database.")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  806982e3-9173-4647-a99f-0c4402b0ceae                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the database for all legal articles or sections relevant to: I was caught jumping a red light     │
│  and the police officer is asking for a 5,000 rupee fine. Is this amount correct?. Provide direct quotes.       │
│  ID: 6972771d-c367-4f35-88e7-f5cefd1dac59                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Task: Search the database for all legal articles or sections relevant to: I was caught jumping a red light     │
│  and the police officer is asking for a 5,000 rupee fine. Is this amount correct?. Provide direct quotes.       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: legal_research_tool                                                                                      │
│  Args: {'query': 'I was caught jumping a red light and the police officer is asking for a 5,000 rupee fine. Is  │
│  this amount correct?'}                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool legal_research_tool executed with result: ...


...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Summarize the strategy into a final answer for the user. Be concise and professional.                          │
│  Agent:                                                                                                         │
│  Legal Advisor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  806982e3-9173-4647-a99f-0c4402b0ceae                                                                           │
│  Final Output: The legal research tool found the following relevant sections and their content:                 │
│                                                                                                                 │
│  Section 177 of the Motor Vehicles Act, 1988:                                                                   │
│  "Any person who disobeys any direction given by or any order made by a traffic policeman, shall be punishable  │
│  with a fine of not exceeding five hundred rupees."                                                             │
│                                                                                                                 │
│  Section 181 of the Motor Vehicles Act, 1988:                                                                   │
│  "A person who does not stop or does not stop within the distance provided, when a police officer stops him,    │
│  will be punishable with fine up to Rs.100/-                                                                    │
│                                                                                                                 │
│  Section 193 of the Motor Vehicles Act, 1988:                                                                   │
│  "Any person who does not follow traffic signs or symbols shall be punishable with a fine of not exceeding One  │
│  Thousand rupees."                                                                                              │
│                                                                                                                 │
│  Section 194 of the Motor Vehicles Act, 1988:                                                                   │
│  "Any person who drives a Motor vehicle without obtaining registration is punishable with fine up to 2,000      │
│  rupees.                                                                                                        │
│                                                                                                                 │
│  Section 199 of the Motor Vehicles Act, 1988:                                                                   │
│  "Any person who drives a Motor vehicle while he is under the influence of intoxicating substance is            │
│  punishable with fine up to 10,000 rupees.                                                                      │
│                                                                                                                 │
│  Section 200 of the Motor Vehicles Act, 1988:                                                                   │
│  "Any person who drives a Motor vehicle without valid driving license is punishable with fine up to  5,000      │
│  rupees.                                                                                                        │
│                                                                                                                 │
│  The maximum fine that can be imposed for jumping a red light is ₹1000 under section 177 or ₹5000 under         │
│  section 200                                          

Evaluation: PASS
Reasoning: Check if the AI corrected the fine amount based on the database.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [9]:
# Question 2
user_input = "I bought a phone last week and it stopped working. The shop owner says 'no returns'. What can I do?"
result = legal_crew.kickoff(inputs={"user_query": user_input})

# Verification Code
print(f"Evaluation: {'PASS' if 'Consumer Protection' in result.raw or 'Consumer' in result.raw else 'FAIL'}")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  806982e3-9173-4647-a99f-0c4402b0ceae                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the database for all legal articles or sections relevant to: I bought a phone last week and it    │
│  stopped working. The shop owner says 'no returns'. What can I do?. Provide direct quotes.                      │
│  ID: 6972771d-c367-4f35-88e7-f5cefd1dac59                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Task: Search the database for all legal articles or sections relevant to: I bought a phone last week and it    │
│  stopped working. The shop owner says 'no returns'. What can I do?. Provide direct quotes.                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: litellm.BadRequestError: GroqException - {"error":{"message":"Failed to call a function. Please adjust  │
│  your prompt. See 'failed_generation' for more                                                                  │
│  details.","type":"invalid_request_error","code":"tool_use_failed","failed_generation":"\u003cfunction=brave_s  │
│  earch\u003e{\"query\":\"Consumer Protection Act 1986 return policy\",\"}\u003c/function\u003e"}}               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name:                                                                                                          │
│  Search the database for all legal articles or sections relevant to: I bought a phone last week and it stopped  │
│  working. The shop owner says 'no returns'. What can I do?. Provide direct quotes.                              │
│  Agent:                                                                                                         │
│  Legal Researcher                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  806982e3-9173-4647-a99f-0c4402b0ceae                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

BadRequestError: litellm.BadRequestError: GroqException - {"error":{"message":"Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.","type":"invalid_request_error","code":"tool_use_failed","failed_generation":"\u003cfunction=brave_search\u003e{\"query\":\"Consumer Protection Act 1986 return policy\",\"}\u003c/function\u003e"}}


In [10]:
# Question 3
user_input = "A female colleague is being bothered by her boss at night over phone calls. Is there a law for women's safety at work?"
result = legal_crew.kickoff(inputs={"user_query": user_input})

# Verification Code
print(f"Evaluation: {'PASS' if 'POSH' in result.raw or 'Sexual Harassment' in result.raw else 'FAIL'}")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the database for all legal articles or sections relevant to: A female colleague is being          │
│  bothered by her boss at night over phone calls. Is there a law for women's safety at work?. Provide direct     │
│  quotes.                                                                                                        │
│  ID: 0e199204-0bca-45df-89b7-a700ad6f6c56                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Task: Search the database for all legal articles or sections relevant to: A female colleague is being          │
│  bothered by her boss at night over phone calls. Is there a law for women's safety at work?. Provide direct     │
│  quotes.                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The previous analysis did not provide a complete picture of the legal provisions related to the question "Is   │
│  there a law for women's safety at work?" Therefore, further analysis is required.                              │
│                                                                                                                 │
│                                                                                                                 │
│  The Indian Penal Code, 1860, under Section 354 D, states that anyone found guilty of voyeurism shall be        │
│  punished with imprisonment for a term that may extend to 3 years and shall also be liable to fine:             │
│                                                                                                                 │
│                                                                                                                 │
│  "(1) The person who captures an image or video of another person without their consent in any private area,    │
│  for sexual gratification or for the purpose of blackmail, will be punished for voyeurism."                     │
│                                                                                                                 │
│                                                                                                                 │
│  The Indian Penal Code, 1860, under Section 354 D (2), further states:                                          │
│                                                                                                                 │
│                                                                                                                 │
│  "The person who captures an image or video of another person in a private area, without their permission, in   │
│  the context of blackmailing that other person by transmitting the images or videos to any person, shall be     │
│  punishable."                                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│  The Information Technology Act, 2000, under Section 66 E, defines as an "illicit traffic in girls" and         │
│  stipulates a punishment for any person who, in the course of providing or obtaining access to any computer     │
│  resource or communication service, captures or publishes any material in which a woman is involved in a        │
│  private setting without her consent, with the aim of creating a sexual sensation for himself or for any other  │
│  man.                                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
│  Section 9 (1) of the Sexual Harassment of Women at Workplace (Prevention, Prohibition and Redressal) Bill of   │
│  2012 specifies:                                                                                                │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Search the database for all legal articles or sections relevant to: A female colleague is being bothered by    │
│  her boss at night over phone calls. Is there a law for women's safety at work?. Provide direct quotes.         │
│  Agent:                                                                                                         │
│  Legal Researcher                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for     │
│  model `llama-3.3-70b-versatile` in organization `org_01kbn58qpqfjgstzyswyfty5g2` service tier `on_demand` on   │
│  tokens per minute (TPM): Limit 12000, Used 6652, Requested 10473. Please try again in 25.625s. Need more       │
│  tokens? Upgrade to Dev Tier today at                                                                           │
│  https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name:                                                                                                          │
│  Summarize the strategy into a final answer for the user. Be concise and professional.                          │
│  Agent:                                                                                                         │
│  Legal Advisor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kbn58qpqfjgstzyswyfty5g2` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 6652, Requested 10473. Please try again in 25.625s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


In [10]:
# Question 4
user_input = "Do daughters have the same right as sons to their father's house in India?"
result = legal_crew.kickoff(inputs={"user_query": user_input})

# Verification Code
print(f"Evaluation: {'PASS' if 'Succession' in result.raw or '2005' in result.raw else 'FAIL'}")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  806982e3-9173-4647-a99f-0c4402b0ceae                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the database for all legal articles or sections relevant to: Do daughters have the same right as  │
│  sons to their father's house in India?. Provide direct quotes.                                                 │
│  ID: 6972771d-c367-4f35-88e7-f5cefd1dac59                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Task: Search the database for all legal articles or sections relevant to: Do daughters have the same right as  │
│  sons to their father's house in India?. Provide direct quotes.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: legal_research_tool                                                                                      │
│  Args: {'query': "Do daughters have the same right as sons to their father's house in India?"}                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool legal_research_tool executed with result: ...


...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name:                                                                                                          │
│  Search the database for all legal articles or sections relevant to: Do daughters have the same right as sons   │
│  to their father's house in India?. Provide direct quotes.                                                      │
│  Agent:                                                                                                         │
│  Legal Researcher                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'task_started' (expected 
'crew_kickoff_started')

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  806982e3-9173-4647-a99f-0c4402b0ceae                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kbn58qpqfjgstzyswyfty5g2` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 3925, Requested 2807. Please try again in 7.32s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


In [11]:
# Question 5
user_input = "Police searched phone/WhatsApp without a warrant for protest evidence. Does this violate Privacy laws?"
result = legal_crew.kickoff(inputs={"user_query": user_input})

# Verification Code
print(f"Evaluation: {'PASS' if 'Privacy' in result.raw or 'Puttaswamy' in result.raw else 'FAIL'}")
print("Check: Did it mention that even for national security, 'procedure established by law' must be followed?")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  806982e3-9173-4647-a99f-0c4402b0ceae                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the database for all legal articles or sections relevant to: Police searched phone/WhatsApp       │
│  without a warrant for protest evidence. Does this violate Privacy laws?. Provide direct quotes.                │
│  ID: 6972771d-c367-4f35-88e7-f5cefd1dac59                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Task: Search the database for all legal articles or sections relevant to: Police searched phone/WhatsApp       │
│  without a warrant for protest evidence. Does this violate Privacy laws?. Provide direct quotes.                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#5) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: legal_research_tool                                                                                      │
│  Args: {'query': 'Police searched phone/WhatsApp without a warrant for protest evidence. Does this violate      │
│  Privacy laws?'}                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool legal_research_tool executed with result: ...


...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name:                                                                                                          │
│  Search the database for all legal articles or sections relevant to: Police searched phone/WhatsApp without a   │
│  warrant for protest evidence. Does this violate Privacy laws?. Provide direct quotes.                          │
│  Agent:                                                                                                         │
│  Legal Researcher                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'task_started' (expected 
'crew_kickoff_started')

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  806982e3-9173-4647-a99f-0c4402b0ceae                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kbn58qpqfjgstzyswyfty5g2` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5682, Requested 3021. Please try again in 27.03s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


In [14]:
# Question 2
user_input = "Brother paralyzed due to wrong injection in a govt hospital. They claim charity immunity. Can we use Consumer Court?"
result = legal_crew.kickoff(inputs={"user_query": user_input})

# Verification Code
print(f"Evaluation: {'PASS' if 'Medical Negligence' in result.raw or 'Deficiency' in result.raw else 'FAIL'}")
print("Check: Did it mention that 'free' services in govt hospitals have specific nuances under the CP Act?")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the database for all legal articles or sections relevant to: Brother paralyzed due to wrong       │
│  injection in a govt hospital. They claim charity immunity. Can we use Consumer Court?. Provide direct quotes.  │
│  ID: 0e199204-0bca-45df-89b7-a700ad6f6c56                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Task: Search the database for all legal articles or sections relevant to: Brother paralyzed due to wrong       │
│  injection in a govt hospital. They claim charity immunity. Can we use Consumer Court?. Provide direct quotes.  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#7) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: legal_research_tool                                                                                      │
│  Args: {'query': 'Charity immunity and Consumer Courts.'}                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool legal_research_tool executed with result: ...


...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name:                                                                                                          │
│  Search the database for all legal articles or sections relevant to: Brother paralyzed due to wrong injection   │
│  in a govt hospital. They claim charity immunity. Can we use Consumer Court?. Provide direct quotes.            │
│  Agent:                                                                                                         │
│  Legal Researcher                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'task_started' (expected 
'crew_kickoff_started')

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kbn58qpqfjgstzyswyfty5g2` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4183, Requested 4278. Please try again in 24.61s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
